# Run inference for EarTTS

In [ ]:
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

import torch
from omegaconf import OmegaConf

import os
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"


# audio tokens for prompt to derive speaker identity
prompt_audio_codes = torch.load("eartts_debug_tokens/prefill_codes.pt").cpu().to(torch.int32)[0]
# subword ids corresponding to the text to synthesize
next_subword_ids = torch.load("eartts_debug_tokens/text_tokens.pt").cpu().to(torch.int32)[0]
print(next_subword_ids.shape)


# load vllm engine
type_str = "float32"
torch_type = getattr(torch, type_str)
engine_args = AsyncEngineArgs(
    model="eartts_vllm_model",
    dtype=type_str,
    max_model_len=768,
    max_num_batched_tokens=768,
    gpu_memory_utilization=0.6,
    skip_tokenizer_init=True,  # Skip tokenizer since we're using embeddings directly
    enable_prefix_caching=False,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=768, skip_sampling=True, guidance_scale=0.5)


generated_tokens = []
for run in range(1):
    request_id = f"request_{run}"

    prompt_len = prompt_audio_codes.shape[0]
    bos_mask = torch.zeros(prompt_len, dtype=torch_type)
    bos_mask[0] = 1.0
    inputs = {
        # dummy tokens
        "prompt_token_ids": [0] * prompt_len,
        # actual inpust to the model in prefill stage
        "custom_inputs": {
            "acoustic_tokens": prompt_audio_codes,
            # masked for prefill stage, so can just provide zeroes
            "text_tokens": torch.zeros(prompt_len, dtype=torch.int32),
            "text_mask": torch.zeros(prompt_len, dtype=torch_type),
            "bos_mask": bos_mask,
        }
    }
    acoustic_tokens_lst = []
    i = 0
    async for output in engine.generate(inputs, sampling_params=sampling_params, request_id=request_id):
        # store predicted acoustic tokens
        acoustic_tokens = output.outputs[0].custom_outputs["acoustic_tokens"]  # T x 31
        step_acoustic_tokens = acoustic_tokens[-1:]  # 1 x 31
        acoustic_tokens_lst.append(step_acoustic_tokens)

        # if previously prepared input was last, break
        if i == next_subword_ids.shape[0] - 1:
            await engine.abort(request_id)
            break

        current_subword_id = next_subword_ids[i:(i+1)]  # (1,)
        new_custom_inputs = {
            "acoustic_tokens": step_acoustic_tokens,
            "text_tokens": current_subword_id,
            "text_mask": torch.ones_like(current_subword_id, dtype=torch_type),
            "bos_mask": torch.zeros_like(current_subword_id, dtype=torch_type),
        }
        await engine.append_request(request_id=request_id, custom_inputs=new_custom_inputs)
        i += 1

    acoustic_tokens_arr = torch.cat(acoustic_tokens_lst, dim=0)
    generated_tokens.append(acoustic_tokens_arr)

for i, tokens in enumerate(generated_tokens):
    torch.save(tokens, f"pred_tokens_{i}.pt")